# U2IE-Net: train from scratch on Google Colab

**Runtime → Change runtime type:** **A100 GPU** (or **G4 GPU**) + **High-RAM**.

U2IE-Net runs Retinex / CLAHE / Gaussian blur with OpenCV on the **CPU** inside every forward pass,
so the number of vCPUs matters as much as the GPU. A full run (50 epochs) is expected to take many hours
and will likely span several Colab sessions: checkpoints are written to Google Drive and `--resume` continues
from the last finished epoch.

### One-time setup: Kaggle token (for the RESIDE-β OTS dataset)
1. On kaggle.com: *Settings → API → Create New Token*.
2. In Colab: click the 🔑 **Secrets** icon in the left sidebar and add `KAGGLE_USERNAME` and `KAGGLE_KEY`
   (or `KAGGLE_API_TOKEN` if Kaggle gave you a single token), then enable *Notebook access*.

### Datasets (all downloaded automatically)
| Folder | Source | Images |
|---|---|---|
| `train/underwater` | UIEB raw-890 (official Google Drive) + RUIE-UIQS (GitHub), minus T400 | ≈ 4,100 |
| `train/haze`, `train/clear` | RESIDE-β OTS (Kaggle `brunobelloni/outdoor-training-set-ots-reside`) | same count, paired |
| `test/T400` | 400 random held-out images from the same underwater pool | 400 |
| `test/UIEB-60`, `test/RUIE-UCCS` | UIEB challenging-60, RUIE-UCCS | 60, 300 |

The paper also uses UICoD (2,286 images) for training, which is not publicly available, so the underwater
set here is ≈ 4,100 images instead of 5,000.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "vCPUs: $(nproc)" && free -h | head -2 && df -h /content | tail -1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# New, self-contained folder: everything that must survive a disconnect lives here
DRIVE_ROOT = '/content/drive/MyDrive/U2IE-Net-Colab'
!mkdir -p "$DRIVE_ROOT"

In [ ]:
REPO_URL = 'https://github.com/nantapongpa-a11y/U2IE-Net.git'
BRANCH = 'U2IE-Net-Colab'

%cd /content
!rm -rf U2IE-Net && git clone -q -b $BRANCH $REPO_URL
%cd /content/U2IE-Net
!pip install -q kornia gdown && pip install -q -U kaggle
!apt-get -qq install -y unrar > /dev/null

## 1. Data

The first run downloads everything (about 13 GB, mostly OTS), builds the training set, and saves it to Drive as
`u2ie_data.zip` (about 2 GB). Later sessions just unzip that file.

In [ ]:
import os
DATA_ZIP = f'{DRIVE_ROOT}/u2ie_data.zip'
RAW, DATA = '/content/raw', '/content/data'

if os.path.exists(DATA_ZIP):
    print('>>> Prepared data found on Drive, unzipping...')
    !mkdir -p $DATA && unzip -q -n "$DATA_ZIP" -d $DATA
else:
    from google.colab import userdata
    for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
    assert 'KAGGLE_KEY' in os.environ or 'KAGGLE_API_TOKEN' in os.environ, \
        'Add your Kaggle token to Colab Secrets first (see the top of this notebook)'

    !mkdir -p $RAW/uieb_raw $RAW/uieb_c60
    # UIEB: official Google Drive links + archive passwords from https://li-chongyi.github.io/proj_benchmark.html
    !gdown 12W_kkblc2Vryb9zHQ6BfGQ_NKUfXYk13 -O $RAW/raw-890.rar
    !gdown 1Ew_r83nXzVk0hlkfuomWqsAIxuq6kaN4 -O $RAW/challenging-60.rar
    for f in ('raw-890.rar', 'challenging-60.rar'):
        assert os.path.exists(f'{RAW}/{f}'), f'{f} download failed (Google Drive quota?) - wait a while and re-run this cell'
    !unrar x -o+ -p1234567 -idq $RAW/raw-890.rar $RAW/uieb_raw/
    !unrar x -o+ -p5678901 -idq $RAW/challenging-60.rar $RAW/uieb_c60/
    n_uieb = sum(len(f) for _, _, f in os.walk(f'{RAW}/uieb_raw'))
    assert n_uieb >= 890, f'UIEB extraction failed ({n_uieb} files)'
    # RUIE: https://github.com/dlut-dimt/Realworld-Underwater-Image-Enhancement-RUIE-Benchmark
    !git clone -q --depth 1 https://github.com/dlut-dimt/Realworld-Underwater-Image-Enhancement-RUIE-Benchmark $RAW/RUIE
    # RESIDE-beta OTS (~12 GB zip; only the selected files are extracted)
    !kaggle datasets download -d brunobelloni/outdoor-training-set-ots-reside -p $RAW
    assert os.path.exists(f'{RAW}/outdoor-training-set-ots-reside.zip'), 'Kaggle download failed - check your token'

    PREP_ARGS = (f'--uieb_raw {RAW}/uieb_raw --ruie_uiqs {RAW}/RUIE/UIQS --exclude {RAW}/RUIE/UCCS {RAW}/uieb_c60 '
                 f'--ots_zip {RAW}/outdoor-training-set-ots-reside.zip --out {DATA} --seed 0')
    !python prepare_data.py {PREP_ARGS}
    assert os.path.exists(f'{DATA}/split.json'), 'prepare_data.py failed - see the error above'

    # test sets, flattened into one folder each
    import shutil
    for src, name in [(f'{RAW}/uieb_c60', 'UIEB-60'), (f'{RAW}/RUIE/UCCS', 'RUIE-UCCS')]:
        os.makedirs(f'{DATA}/test/{name}', exist_ok=True)
        for root, _, files in os.walk(src):
            for fn in files:
                if fn.lower().endswith(('.png', '.jpg', '.jpeg')):
                    shutil.copy(os.path.join(root, fn), f'{DATA}/test/{name}/{fn}')

    !cd $DATA && zip -q -r -0 /content/u2ie_data.zip . && cp /content/u2ie_data.zip "$DATA_ZIP"
    !rm -rf $RAW /content/u2ie_data.zip

!for d in train/underwater train/haze train/clear test/T400 test/UIEB-60 test/RUIE-UCCS; do echo "$$d: $(ls $DATA/$$d | wc -l)"; done

## 2. Training

Hyper-parameters follow the paper (Adam, lr 0.0005, batch 8, 50 epochs, 256×256).
After a minute, read `s/it` from the progress bar: **total hours ≈ s/it × iterations-per-epoch × 50 / 3600**.

**If Colab disconnects:** reconnect, run the setup cells and the data cell again, then this cell. `--resume` picks up from the last finished epoch.

In [ ]:
import os
NUM_WORKERS = max(1, os.cpu_count() - 1)
OUT = f'{DRIVE_ROOT}/OpenData'

TRAIN_ARGS = ' '.join([
    f'--u_path {DATA}/train/underwater --h_path {DATA}/train/haze --g_path {DATA}/train/clear',
    f'--output_dir "{OUT}"',
    '--epochs 50 --batch_size 8 --lr 0.0005 --save_interval 10',
    f'--num_workers {NUM_WORKERS} --resume',
])
!python train.py {TRAIN_ARGS}

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage(f'{OUT}/lossdata/training_loss_curve.png'))

## 3. Enhance the test sets with the trained model

In [ ]:
CKPT = f'{OUT}/checkpoints/u2ie_latest.pth'
for name in ['T400', 'UIEB-60', 'RUIE-UCCS']:
    !python test.py --input_dir "{DATA}/test/{name}" --output_dir "{DRIVE_ROOT}/Results/{name}" --checkpoint "{CKPT}" --batch_size 16

In [ ]:
# Side-by-side preview: input (top) vs enhanced (bottom)
import os, matplotlib.pyplot as plt
from PIL import Image

name = 'UIEB-60'
src, dst = f'{DATA}/test/{name}', f'{DRIVE_ROOT}/Results/{name}'
files = sorted(os.listdir(dst))[:6]
fig, ax = plt.subplots(2, len(files), figsize=(3 * len(files), 6))
for i, n in enumerate(files):
    ax[0, i].imshow(Image.open(os.path.join(src, n)).resize((256, 256)))
    ax[1, i].imshow(Image.open(os.path.join(dst, n)))
    ax[0, i].set_title(n[:18], fontsize=8)
    for a in ax[:, i]: a.axis('off')
plt.tight_layout(); plt.show()